In [12]:
from dotenv import load_dotenv
import json
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [2]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [3]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [4]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1400

In [5]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [6]:
def rag(self, query):
    search_results = self.search(query)
    prompt = self.build_prompt(query, search_results)
    answer = self.llm(prompt)
    return answer

In [7]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [9]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [10]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [13]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment registration new student"}
function_call: search {"query":"course join late enrollment can I join discovered the course"}
function_call: search {"query":"FAQ join the course register after course starts discover course"}


In [18]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [20]:
#agent_loop(instructions, "How do I run Olama locally?")

In [21]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late enrollment discovered course can I still join"}
function_call: search {"query":"course enrollment late join discovered course FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join. The course materials are available, and you can start whenever you want.

One important note: if you want a certificate, you need to submit your project while submissions are still being accepted.

If you’d like, I can also help you with:
- how to start the course,
- whether you can still do homework,
- or what’s needed for the certificate.

Anything else you want to explore?


'Yes — you can still join. The course materials are available, and you can start whenever you want.\n\nOne important note: if you want a certificate, you need to submit your project while submissions are still being accepted.\n\nIf you’d like, I can also help you with:\n- how to start the course,\n- whether you can still do homework,\n- or what’s needed for the certificate.\n\nAnything else you want to explore?'

In [22]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment registration late join FAQ"}
iteration #2...
function_call: search {"query":"certificate submit project while accepting submissions join course late self paced live cohort FAQ project submission peer reviews"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, the key thing is to submit your capstone project while submissions are still open, and complete the required peer reviews during the live cohort window. You can still go through the material in self-paced mode, but the certificate is only available through the live cohort process.

If you'd like, I can also help with:
- how to get the certificate,
- how project submission works,
- or whether you can still follow the course self-paced.


"Yes — you can still join the course.\n\nIf you want a certificate, the key thing is to submit your capstone project while submissions are still open, and complete the required peer reviews during the live cohort window. You can still go through the material in self-paced mode, but the certificate is only available through the live cohort process.\n\nIf you'd like, I can also help with:\n- how to get the certificate,\n- how project submission works,\n- or whether you can still follow the course self-paced."

In [24]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening queen gambit FAQ"}
iteration #3...
ASSISTANT:
I couldn’t find a course FAQ entry for “queen gambit,” so it’s likely off-topic for this course.

If you meant **“Queen’s Gambit”** in chess, that’s a chess opening — but I can’t provide non-course explanations here unless it appears in the course FAQ.

If you want, I can help you look for a **course-related** topic instead. Are there other areas you want to explore?


'I couldn’t find a course FAQ entry for “queen gambit,” so it’s likely off-topic for this course.\n\nIf you meant **“Queen’s Gambit”** in chess, that’s a chess opening — but I can’t provide non-course explanations here unless it appears in the course FAQ.\n\nIf you want, I can help you look for a **course-related** topic instead. Are there other areas you want to explore?'